In [2]:
# We are building a tiny semantic search engine.
#mental model

# Wikipedia dataset	gives us text passages
# SentenceTransformer	converts passages into embeddings
# FAISS	stores/searches embeddings
# corpus.pkl	stores original text passages
# wiki.faiss	stores FAISS vector index

# Notes:

# FAISS does not create embeddings.
# SentenceTransformer creates embeddings.
# FAISS stores/searches embeddings.

!pip install faiss-cpu

import os
import pickle #to save/load python obkjects. ex.corpus
import faiss
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

print ("loading dataset")

#load_dataset loads ds from hugging face.
ds = load_dataset ("rag-datasets/rag-mini-wikipedia", #ds name
                    "text-corpus")["passages"] #passages is the split in the ds

print (ds)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 74.1 MB/s eta 0:00:00
loading dataset


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/719 [00:00<?, ?B/s]

data/passages.parquet/part.0.parquet:   0%|          | 0.00/797k [00:00<?, ?B/s]

Generating passages split:   0%|          | 0/3200 [00:00<?, ? examples/s]

Dataset({
    features: ['passage', 'id'],
    num_rows: 3200
})


In [5]:
#we get the passage col, first 20k
corpus = ds["passage"][:20000]

In [7]:
model = SentenceTransformer("all-MiniLM-L6-v2") #load embeddding model

emb = model.encode( #turn text into embedding vectors
    corpus,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)
emb = np.array (emb, dtype="float32") #convert to faiss format

print("Embedding shape:", emb.shape) #ex. expected: Embedding shape: (20000, 384)

#create faiss index
index = faiss.IndexFlatIP(emb.shape[1])

index.add (emb)
#save faisss index to disk
faiss.write_index (index, "wiki.faiss")

#This saves the original text passages.
with open ("corpus.pkl", "wb") as f:
  pickle.dump (corpus,f)
print("Saved FAISS index to day05/wiki.faiss")
print("Saved corpus to day05/corpus.pkl")

#embed query
query = model.encode(
    ["what are fossil fuels?"],
    normalize_embeddings=True,
).astype("float32")

#search for query in faiss
D, I = index.search(query, k=5)

print("\nSearch results:")
for rank, (score, idx) in enumerate(zip(D[0], I[0]), start=1):
    print(f"\n{rank}. score={score:.3f}")
    print(corpus[idx][:500])



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/50 [00:00<?, ?it/s]

Embedding shape: (3200, 384)
Saved FAISS index to day05/wiki.faiss
Saved corpus to day05/corpus.pkl

Search results:

1. score=0.488
Most of the energy is produced from fossil fuels, mainly coal and oil. Fossil fuels are, however, all imported, because Finland doesn't have any fossil fuel sources, unlike neighboring Norway with oil and Estonia with oil shale. Nevertheless, Finland fares exceptionally well with renewable energy: 25 percent of energy is renewable, which is high compared to the EU average 10 percent. About one fifth of all the energy consumed in Finland is wood-based. This is not a remnant of old ages: the pulp 

2. score=0.413
Canada is one of the few developed nations that is a net exporter of energy.  Atlantic Canada has vast offshore deposits of natural gas and large oil and gas resources are centred in Alberta. The vast Athabasca Tar Sands give Canada the world's second largest reserves of oil behind Saudi Arabia.    In Quebec, British Columbia, Newfoundland & Labrad

In [8]:
#5.5

import pickle

import faiss
import gradio as gr
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
index = faiss.read_index("wiki.faiss")

with open ("corpus.pkl", "rb") as f:
    corpus = pickle.load (f)

def search(query, k=5):
    q = model.encode(
        [query],
        normalize_embeddings=True,
    ).astype("float32")

    D, I = index.search(q, k)

    results = []

    #Why D[0] and I[0]? Because FAISS returns results in a 2D structure, even if you search with only one query
    for score, idx in zip(D[0], I[0]):
        passage = corpus[idx][:400]
        results.append(f"**Score {score:.3f}**\n\n{passage}...")

        return "\n\n---\n\n".join(results)

gr.Interface(
    fn=search, #When user clicks submit, call the search function.
    inputs =[ #creates 2 ui components
        gr.Textbox(label="query"),
        gr.Slider(1, 20, value=5, step=1, label="Top-K"),
    ],
    outputs = [gr.Markdown (label="Results")],
    title="Day 5 Semantic Search",

).launch(share=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f3b7ab5653d0491599.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
